In [ ]:
import math
import numpy as np
from scipy.optimize import fsolve

def calculate_theoretical_trajectory(k_spring, x_spring, mass, theta_deg, h, drag_c, target_x=None):
    """
    용수철 발사 장치 및 공기 저항(선형 항력)을 고려한 이론적 궤적 및 최종 낙하 거리를 계산합니다.
    
    [입력 파라미터 단위]
    - k_spring : 용수철 탄성 계수 (N/m)
    - x_spring : 용수철 압축 거리 (m)
    - mass     : 공의 질량 (kg)
    - theta_deg: 발사 각도 (Degree)
    - h        : 초기 발사 높이 (m)
    - drag_c   : 공기 저항 계수 (kg/s)
    - target_x : 특정 높이를 알고 싶은 수평 거리 x (m), 기본값은 None
    """
    g = 9.81
    theta = math.radians(theta_deg)
    cos_theta = math.cos(theta)
    tan_theta = math.tan(theta)
    
    # 1. 역학적 에너지 보존 법칙을 이용한 초기 속도(v0) 계산
    v0 = x_spring * math.sqrt(k_spring / mass)
    v0x = v0 * cos_theta
    v0z = v0 * math.sin(theta)
    
    # 공기 저항이 없는 진공 상태(c=0)일 때의 예외 처리
    if drag_c == 0:
        t_land_vac = (v0z + math.sqrt(v0z**2 + 2 * g * h)) / g
        x_max_vac = v0x * t_land_vac
        
        z_at_target = None
        if target_x is not None:
            # 진공 상태 포물선 방정식: z(x) = h + x*tan(theta) - (g*x^2)/(2*v0^2*cos^2(theta))
            z_at_target = h + target_x * tan_theta - (g * (target_x**2)) / (2 * (v0**2) * (cos_theta**2))
            
        return {
            "initial_velocity_v0": round(v0, 3),
            "max_landing_distance": round(x_max_vac, 3),
            "z_at_target_x": round(z_at_target, 3) if z_at_target is not None else None
        }

    # 2. 공기 저항이 있을 때 (c > 0) z(t) = 0을 만족하는 비행 시간 t_land 구하기
    # z(t) 수식 정의
    z_t_func = lambda t: h + (mass / drag_c) * (v0z + (mass * g) / drag_c) * (1 - np.exp(-drag_c * t / mass)) - (mass * g * t) / drag_c
    
    # 수치해석 풀이를 위한 초기 비행 시간 추정값 결정 (진공 상태 기준)
    t_guess = (v0z + math.sqrt(v0z**2 + 2 * g * h)) / g
    t_land = fsolve(z_t_func, t_guess)[0]
    
    # 3. 비행 시간을 수평 위치 방정식 x(t)에 대입하여 최종 이론적 낙하 거리(x_max) 도출
    x_max = (mass * v0x / drag_c) * (1 - math.exp(-drag_c * t_land / mass))
    
    # 4. 사용자 요청 시, 소거식 z(x)를 이용해 특정 수水平 거리 x에서의 높이(z) 계산
    z_at_target = None
    if target_x is not None:
        # 이론적으로 도출한 z(x) 통합 궤적 방정식
        inside_ln = 1 - (drag_c * target_x) / (mass * v0 * cos_theta)
        
        if inside_ln <= 0:
            # 수평 도달 한계치(물리적 장벽)를 넘어선 경우 고정 낙하 처리
            z_at_target = float('-inf')
        else:
            term1 = h
            term2 = (tan_theta + (mass * g) / (drag_c * v0 * cos_theta)) * target_x
            term3 = ((mass**2) * g / (drag_c**2)) * math.log(inside_ln)
            z_at_target = term1 + term2 + term3

    return {
        "initial_velocity_v0": round(v0, 3),
        "max_landing_distance": round(x_max, 3),
        "z_at_target_x": round(z_at_target, 3) if z_at_target is not None else None
    }


# =====================================================================
# 🔍 실제 실험 데이터 연산 검증 (System ID 결과 대입)
# =====================================================================

# 공통 환경 변수
K_SPRING = 320.0     # 용수철 탄성 계수 (N/m)
X_SPRING = 0.05      # 용수철 압축 거리 5cm (m)
LAUNCH_ANGLE = 6.0   # 실제 최적 발사 각도 (Degree)
LAUNCH_HEIGHT = 1.0  # 초기 발사 높이 1m

print("--- [Case A: 3g 공 실험 환경 이론값 검증] ---")
result_3g = calculate_theoretical_trajectory(
    k_spring=K_SPRING, x_spring=X_SPRING, mass=0.003, 
    theta_deg=LAUNCH_ANGLE, h=LAUNCH_HEIGHT, drag_c=0.0211, target_x=0.5
)
print(f"▶ 계산된 초기 속도 (v0)  : {result_3g['initial_velocity_v0']} m/s")
print(f"▶ 이론적 최종 낙하 거리 : {result_3g['max_landing_distance']} m (실제 측정값: 2.25m 내외)")
print(f"▶ 전방 0.5m 지점에서의 높이: {result_3g['z_at_target_x']} m\n")


print("--- [Case B: 27g 공 실험 환경 이론값 검증] ---")
result_27g = calculate_theoretical_trajectory(
    k_spring=K_SPRING, x_spring=X_SPRING, mass=0.027, 
    theta_deg=LAUNCH_ANGLE, h=LAUNCH_HEIGHT, drag_c=0.0438, target_x=0.5
)
print(f"▶ 계산된 초기 속도 (v0)  : {result_3g['initial_velocity_v0']} m/s")
print(f"▶ 이론적 최종 낙하 거리 : {result_27g['max_landing_distance']} m (실제 측정값: 1.30m 내외)")
print(f"▶ 전방 0.5m 지점에서의 높이: {result_27g['z_at_target_x']} m")